# 04 — Generate routing features

Produce annual nearest-infrastructure features, narrow grid-quarter population and lagged-total-firm accessibility, firm-quarter same-Fachgruppe accessibility, and the separate long grid-quarter-Fachgruppe table.

In [ ]:
from __future__ import annotations

from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import gc
import math
import numpy as np
import subprocess
import sys
import time

import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from shapely.geometry import box, shape
from tqdm.auto import tqdm


PROJECT_DIR = Path(r"D:\CO2_Masterarbeit\CO2_Masterarbeit")
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
POI_DIR = PROJECT_DIR / "TOOLS" / "pois"
ROUTING_DIR = PROJECT_DIR / "TOOLS" / "routing"

ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
FIRMS_PATH = ANAL_DATA / "firms_assigned_100m.geoparquet"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
RASTER_PATH = ANAL_DATA / "raster_100m_styria.geoparquet"
FEATURE_ROOT = ROUTING_DATA / "features"
STATUS_DIR = ROUTING_DATA / "status"
ROUTING_STATUS_PATH = STATUS_DIR / "routing_feature_status.csv"

sys.path.insert(0, str(ROUTING_DIR))
from routing_utils import ACCESS_MINUTES, fachgruppe_access_columns, fachgruppe_ids, fachgruppe_stock_columns, main_access_columns

VALHALLA_IMAGE = "ghcr.io/valhalla/valhalla-scripted:latest"
VALHALLA_URL = "http://localhost:8002"
VALHALLA_CPUS = 16
WSL_EXE = r"C:\Windows\System32\wsl.exe"
WSL_DISTRO = "Ubuntu"
WSL_PROJECT_ROOT = "$HOME/gruendungsanalyse"
WSL_GRAPH_ROOT = f"{WSL_PROJECT_ROOT}/data/routing/valhalla_graphs"
START_YEAR, END_YEAR = 2015, 2025
FACHGRUPPE_IDS = fachgruppe_ids(PANEL_PATH)
if len(FACHGRUPPE_IDS) != 95:
    raise ValueError(f"Expected 95 Fachgruppen, found {len(FACHGRUPPE_IDS)}")

FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
STATUS_DIR.mkdir(parents=True, exist_ok=True)

## Run configuration

Use `RUN_MODE = "smoke"` for a non-writing 2025 API check, then `RUN_MODE = "full"` for supported outputs. `dry-run` only prints the work.

In [ ]:
RUN_MODE = "dry-run"  # "dry-run", "smoke", or "full"
YEARS_TO_RUN = [2025]
RUN_NEAREST_INFRASTRUCTURE = True
RUN_ACCESSIBILITY = True
DESTINATION_TYPES = ["rail_station", "motorway_exit", "regional_centre", "urban_centre", "higher_education"]
PT_WALK_POI_TYPE = "pt_stop"
PT_WALK_MAX_MINUTES = 5
PT_WALK_MAX_SECONDS = 300
PT_WALK_EUCLIDEAN_PREFILTER_M = 1000
PT_NEAREST_EUCLIDEAN_PREFILTER_M = 2000
PT_NEAREST_DESTINATION_STEP_SHARE = 0.002
MAX_DESTINATIONS_PER_TYPE = None
VALHALLA_MAX_MATRIX_PAIRS = 2500
ORIGIN_CHUNK_SIZE = 24
DESTINATION_CHUNK_SIZE = 100
MAX_CONCURRENT_REQUESTS = 24
NEAREST_INFRA_EUCLIDEAN_PREFILTER_M = 10_000
NEAREST_INFRA_DESTINATION_STEP_SHARE = 0.10
NEAREST_INFRA_TYPE_WORKERS = len(DESTINATION_TYPES)
NEAREST_INFRA_ORIGIN_CHUNK_WORKERS = max(1, math.ceil(MAX_CONCURRENT_REQUESTS / NEAREST_INFRA_TYPE_WORKERS))
NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS = NEAREST_INFRA_ORIGIN_CHUNK_WORKERS * 4
PT_WALK_ORIGIN_CHUNK_WORKERS = MAX_CONCURRENT_REQUESTS
PT_WALK_MAX_IN_FLIGHT_ORIGIN_CHUNKS = MAX_CONCURRENT_REQUESTS * 4
POPULATION_ACCESS_COSTING = "auto"
POPULATION_ACCESS_CONTOURS_MIN = list(ACCESS_MINUTES)
POPULATION_ACCESS_POLYGONS = True
POPULATION_ACCESS_DENOISE = 0
POPULATION_ACCESS_GENERALIZE_M = 0
POPULATION_ACCESS_ORIGIN_WORKERS = 24
POPULATION_ACCESS_MAX_IN_FLIGHT_ORIGINS = 32
POPULATION_ACCESS_FLUSH_ORIGINS = 500
POPULATION_ACCESS_REQUEST_TIMEOUT_SECONDS = 180
POPULATION_ACCESS_REQUEST_RETRIES = 3
MAX_WAIT_MINUTES = 30
MAX_ACTIVE_CELLS = None

## WSL / Docker Helpers

In [ ]:
def run_local(command: list[str], check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(command, check=check, text=True, capture_output=capture_output)


def wsl_base_command() -> list[str]:
    if WSL_DISTRO:
        return [WSL_EXE, "-d", WSL_DISTRO, "--"]
    return [WSL_EXE, "--"]


def run_wsl(command: str, check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return run_local([*wsl_base_command(), "bash", "-lc", command], check=check, capture_output=capture_output)


def require_wsl_distribution() -> None:
    distro_list = run_local([WSL_EXE, "-l", "-q"], check=False)
    available_distros = [line.strip().replace("\x00", "") for line in distro_list.stdout.splitlines() if line.strip().replace("\x00", "")]
    if WSL_DISTRO and available_distros and WSL_DISTRO not in available_distros:
        raise RuntimeError(
            f"Configured WSL_DISTRO={WSL_DISTRO!r}, but PowerShell reports these WSL distros: {available_distros}. "
            "Set WSL_DISTRO to the exact name from `wsl -l -v`."
        )
    test = run_wsl("printf ok", check=False)
    if test.returncode != 0:
        raise RuntimeError(
            "Could not start the configured WSL distribution. Run `wsl -l -v` in PowerShell "
            "and set WSL_DISTRO in this notebook to the exact distro name.\n\n"
            f"stdout:\n{test.stdout}\n\nstderr:\n{test.stderr}"
        )


def quote_bash(value: str) -> str:
    return "'" + value.replace("'", "'\\''") + "'"


def quote_wsl_path(value: str) -> str:
    if value.startswith("$HOME/"):
        return "$HOME/" + quote_bash(value.removeprefix("$HOME/"))
    return quote_bash(value)


def wsl_graph_dir(year: int) -> str:
    return f"{WSL_GRAPH_ROOT}/{year}"


def wsl_manifest_path(year: int) -> str:
    return f"{wsl_graph_dir(year)}/build_manifest.json"


def container_name(year: int) -> str:
    return f"co2-valhalla-routing-{year}"


def graph_manifest_exists(year: int) -> bool:
    return run_wsl(f"test -f {quote_wsl_path(wsl_manifest_path(year))}", check=False).returncode == 0


def start_valhalla_container(year: int) -> None:
    if not graph_manifest_exists(year):
        raise FileNotFoundError(f"Missing graph manifest in WSL: {wsl_manifest_path(year)}")
    name = container_name(year)
    graph_dir = wsl_graph_dir(year)
    command = " && ".join([
        f"docker rm -f {quote_bash(name)} >/dev/null 2>&1 || true",
        "docker run -d "
        f"--name {quote_bash(name)} "
        f"--cpus {VALHALLA_CPUS} "
        "-p 8002:8002 "
        "-e build_admins=True "
        "-e build_time_zones=True "
        "-e build_tar=True "
        "-e serve_tiles=True "
        f"-v {quote_wsl_path(graph_dir)}:/custom_files "
        f"{quote_bash(VALHALLA_IMAGE)}",
    ])
    run_wsl(command)


def stop_valhalla_container(year: int) -> None:
    run_wsl(f"docker rm -f {quote_bash(container_name(year))} >/dev/null 2>&1 || true", check=False)


def wait_until_valhalla_ready(year: int, max_wait_minutes: int = MAX_WAIT_MINUTES) -> dict:
    deadline = time.time() + max_wait_minutes * 60
    last_error = None
    while time.time() < deadline:
        try:
            summary = valhalla_test_route()
            print(f"Valhalla ready for {year}: {summary}")
            return summary
        except Exception as error:
            last_error = error
            time.sleep(10)
    logs = run_wsl(f"docker logs --tail 80 {quote_bash(container_name(year))}", check=False).stdout
    raise TimeoutError(f"Valhalla did not become ready for {year}. Last error: {last_error}\n\nContainer logs:\n{logs}")

## Routing Helpers

In [ ]:
def output_paths(year: int) -> dict:
    feature_dir = FEATURE_ROOT / str(year)
    feature_dir.mkdir(parents=True, exist_ok=True)
    return {
        "nearest": feature_dir / "nearest_infrastructure_100m.parquet",
        "nearest_partial": feature_dir / "nearest_infrastructure_100m.partial.parquet",
        "potentials": feature_dir / "accessibility_potentials_100m.parquet",
        "potentials_parts": feature_dir / ".accessibility_parts",
        "fachgruppe_accessibility": feature_dir / "fachgruppe_accessibility_quarter_100m.parquet",
        "firm_accessibility": feature_dir / "firm_accessibility_quarter_100m.parquet",
    }


def valhalla_test_route() -> dict:
    payload = {
        "locations": [
            {"lat": 47.0707, "lon": 15.4395},
            {"lat": 47.0580, "lon": 15.4630},
        ],
        "costing": "auto",
        "directions_options": {"units": "kilometers"},
    }
    response = requests.post(f"{VALHALLA_URL}/route", json=payload, timeout=30)
    response.raise_for_status()
    summary = response.json()["trip"]["summary"]
    if summary.get("time", 0) <= 0:
        raise RuntimeError(f"Valhalla responded but returned an invalid route summary: {summary}")
    return summary


def route_matrix(origins: pd.DataFrame, destinations: gpd.GeoDataFrame, costing: str = "auto") -> list[list[dict]]:
    matrix_pairs = len(origins) * len(destinations)
    if matrix_pairs > VALHALLA_MAX_MATRIX_PAIRS:
        raise ValueError(
            f"Matrix request has {matrix_pairs:,} pairs "
            f"({len(origins):,} origins * {len(destinations):,} destinations), "
            f"above Valhalla limit {VALHALLA_MAX_MATRIX_PAIRS:,}. "
            "Lower ORIGIN_CHUNK_SIZE or DESTINATION_CHUNK_SIZE."
        )
    payload = {
        "sources": [
            {"lat": float(row.lat), "lon": float(row.lon)}
            for row in origins.itertuples(index=False)
        ],
        "targets": [
            {"lat": float(row.lat), "lon": float(row.lon)}
            for row in destinations.itertuples(index=False)
        ],
        "costing": costing,
    }
    response = requests.post(f"{VALHALLA_URL}/sources_to_targets", json=payload, timeout=180)
    if not response.ok:
        raise RuntimeError(
            f"Valhalla matrix request failed with HTTP {response.status_code}: {response.text[:1000]}"
        )
    data = response.json()
    if "sources_to_targets" not in data:
        raise RuntimeError(f"Unexpected Valhalla matrix response keys: {sorted(data.keys())}")
    return data["sources_to_targets"]


def route_matrix_chunk(
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    costing: str = "auto",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame, list[list[dict]]]:
    return origins, destinations, route_matrix(origins, destinations, costing=costing)


def finite_number(value: object) -> bool:
    return isinstance(value, (int, float)) and math.isfinite(value)


def normalize_destinations(pois: gpd.GeoDataFrame, poi_type: str) -> gpd.GeoDataFrame:
    selected = pois[pois["poi_type"] == poi_type].copy()
    if selected.empty:
        return selected
    selected = selected.to_crs("EPSG:4326")
    selected["lon"] = selected.geometry.x
    selected["lat"] = selected.geometry.y
    if "poi_id" not in selected.columns:
        selected["poi_id"] = poi_type + "_" + selected.index.astype(str)
    if MAX_DESTINATIONS_PER_TYPE is not None:
        selected = selected.head(MAX_DESTINATIONS_PER_TYPE).copy()
    return selected.reset_index(drop=True)


def nearest_infrastructure_stage_size(total_destinations: int, step_share: float | None = None) -> int:
    share = NEAREST_INFRA_DESTINATION_STEP_SHARE if step_share is None else step_share
    return max(1, math.ceil(total_destinations * share))


def update_best_routes(
    best_by_origin: dict,
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    matrix: list[list[dict]],
    eligible_destination_indices_by_origin: dict | None = None,
) -> None:
    for origin_row, result_row in zip(origins.itertuples(index=False), matrix):
        current_best = best_by_origin.get(origin_row.grid_id)
        eligible_indices = None
        if eligible_destination_indices_by_origin is not None:
            eligible_indices = eligible_destination_indices_by_origin.get(origin_row.grid_id)
        for destination_index, result in enumerate(result_row):
            destination_global_index = int(destinations.index[destination_index])
            if eligible_indices is not None and destination_global_index not in eligible_indices:
                continue
            if result.get("status", 0) != 0:
                continue
            result_time = result.get("time")
            result_distance = result.get("distance")
            if not finite_number(result_time) or not finite_number(result_distance):
                continue
            current_best_time = math.inf if current_best is None else current_best["result"].get("time", math.inf)
            if result_time < current_best_time:
                current_best = {
                    "result": result,
                    "destination": destinations.iloc[destination_index],
                }
        best_by_origin[origin_row.grid_id] = current_best


def best_routes_to_records(best_by_origin: dict, poi_type: str) -> pd.DataFrame:
    records = []
    for grid_id, best_payload in best_by_origin.items():
        if best_payload is None:
            records.append({
                "grid_id": grid_id,
                f"tt_{poi_type}_min": pd.NA,
                f"km_{poi_type}": pd.NA,
                f"nearest_{poi_type}_id": pd.NA,
                f"routing_status_{poi_type}": "unroutable",
            })
            continue
        best = best_payload["result"]
        destination = best_payload["destination"]
        records.append({
            "grid_id": grid_id,
            f"tt_{poi_type}_min": float(best.get("time")) / 60,
            f"km_{poi_type}": best.get("distance"),
            f"nearest_{poi_type}_id": destination.get("poi_id"),
            f"routing_status_{poi_type}": "ok",
        })
    columns = [
        "grid_id",
        f"tt_{poi_type}_min",
        f"km_{poi_type}",
        f"nearest_{poi_type}_id",
        f"routing_status_{poi_type}",
    ]
    return pd.DataFrame(records, columns=columns)


def pt_walk_default_records(grid_ids: list[str] | pd.Series) -> pd.DataFrame:
    grid_id_list = list(grid_ids)
    return pd.DataFrame({
        "grid_id": grid_id_list,
        "has_pt_stop_5min_walk": [False] * len(grid_id_list),
        "pt_departures_5min_walk": [0.0] * len(grid_id_list),
    })


def nearest_for_origin_chunk(
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    destination_x: np.ndarray,
    destination_y: np.ndarray,
    poi_type: str,
    costing: str = "auto",
    euclidean_prefilter_m: int = NEAREST_INFRA_EUCLIDEAN_PREFILTER_M,
    destination_step_share: float | None = None,
) -> tuple[pd.DataFrame, int]:
    best_by_origin = {grid_id: None for grid_id in origins["grid_id"]}
    total_destinations = len(destinations)
    stage_size = nearest_infrastructure_stage_size(total_destinations, destination_step_share)
    radius_sq = euclidean_prefilter_m ** 2
    completed_requests = 0

    origin_grid_ids = origins["grid_id"].tolist()
    origin_x = origins["centroid_x_3035"].to_numpy(dtype=float)
    origin_y = origins["centroid_y_3035"].to_numpy(dtype=float)
    distance_sq = (origin_x[:, None] - destination_x[None, :]) ** 2 + (origin_y[:, None] - destination_y[None, :]) ** 2
    ordered_destination_indices = np.argsort(distance_sq, axis=1)
    initial_limits = np.maximum(stage_size, (distance_sq <= radius_sq).sum(axis=1).astype(int))
    current_limits = np.clip(initial_limits, 1, total_destinations)
    previous_limits = np.zeros(len(origins), dtype=int)
    unresolved_positions = np.arange(len(origins), dtype=int)

    while len(unresolved_positions) > 0:
        eligible_by_origin = {}
        pending_positions = []
        requested_destination_indices = set()
        for row_position in unresolved_positions:
            start = int(previous_limits[row_position])
            stop = int(current_limits[row_position])
            if start >= stop:
                continue
            candidate_indices = {
                int(index)
                for index in ordered_destination_indices[row_position, start:stop]
            }
            if not candidate_indices:
                continue
            pending_positions.append(int(row_position))
            eligible_by_origin[origin_grid_ids[row_position]] = candidate_indices
            requested_destination_indices.update(candidate_indices)

        if not requested_destination_indices:
            break

        pending_origins = origins.iloc[pending_positions].copy()
        destination_indices = sorted(requested_destination_indices)
        for destination_start in range(0, len(destination_indices), DESTINATION_CHUNK_SIZE):
            chunk_indices = destination_indices[destination_start:destination_start + DESTINATION_CHUNK_SIZE]
            destination_chunk = destinations.iloc[chunk_indices].copy()
            matrix = route_matrix(pending_origins, destination_chunk, costing=costing)
            update_best_routes(
                best_by_origin,
                pending_origins,
                destination_chunk,
                matrix,
                eligible_destination_indices_by_origin=eligible_by_origin,
            )
            completed_requests += 1

        previous_limits[pending_positions] = current_limits[pending_positions]
        next_unresolved_positions = [
            int(row_position)
            for row_position in unresolved_positions
            if best_by_origin[origin_grid_ids[row_position]] is None and current_limits[row_position] < total_destinations
        ]
        if not next_unresolved_positions:
            break
        current_limits[next_unresolved_positions] = np.minimum(
            total_destinations,
            current_limits[next_unresolved_positions] + stage_size,
        )
        unresolved_positions = np.array(next_unresolved_positions, dtype=int)

    return best_routes_to_records(best_by_origin, poi_type), completed_requests


def nearest_for_type(
    active_cells: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    poi_type: str,
    progress_position: int | None = None,
    costing: str = "auto",
    euclidean_prefilter_m: int = NEAREST_INFRA_EUCLIDEAN_PREFILTER_M,
    destination_step_share: float | None = None,
) -> pd.DataFrame:
    destinations_3035 = destinations.to_crs("EPSG:3035")
    destination_x = destinations_3035.geometry.x.to_numpy(dtype=float)
    destination_y = destinations_3035.geometry.y.to_numpy(dtype=float)
    chunk_starts = list(range(0, len(active_cells), ORIGIN_CHUNK_SIZE))
    completed_requests = 0
    chunk_results = {}
    progress_kwargs = {
        "total": len(active_cells),
        "desc": f"Nearest infrastructure: {poi_type}",
        "unit": "origin",
        "leave": True,
    }
    if progress_position is not None:
        progress_kwargs["position"] = progress_position
    progress_bar = tqdm(**progress_kwargs)

    def submit_next(executor, next_chunk_index: int, futures: dict) -> int:
        if next_chunk_index >= len(chunk_starts):
            return next_chunk_index
        chunk_start = chunk_starts[next_chunk_index]
        origins = active_cells.iloc[chunk_start:chunk_start + ORIGIN_CHUNK_SIZE].copy()
        future = executor.submit(
            nearest_for_origin_chunk,
            origins,
            destinations,
            destination_x,
            destination_y,
            poi_type,
            costing,
            euclidean_prefilter_m,
            destination_step_share,
        )
        futures[future] = chunk_start
        return next_chunk_index + 1

    try:
        with ThreadPoolExecutor(max_workers=NEAREST_INFRA_ORIGIN_CHUNK_WORKERS) as executor:
            futures = {}
            next_chunk_index = 0
            while next_chunk_index < len(chunk_starts) and len(futures) < NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                next_chunk_index = submit_next(executor, next_chunk_index, futures)

            while futures:
                for completed in as_completed(list(futures)):
                    chunk_start = futures.pop(completed)
                    nearest_chunk, request_count = completed.result()
                    chunk_results[chunk_start] = nearest_chunk
                    completed_requests += request_count
                    progress_bar.update(len(nearest_chunk))
                    progress_bar.set_postfix_str(f"{completed_requests:,} matrix requests")
                    while next_chunk_index < len(chunk_starts) and len(futures) < NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                        next_chunk_index = submit_next(executor, next_chunk_index, futures)
                    break
    finally:
        progress_bar.close()

    if not chunk_results:
        return best_routes_to_records({}, poi_type)
    return pd.concat([chunk_results[chunk_start] for chunk_start in chunk_starts], ignore_index=True)


def pt_walk_for_origin_chunk(
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    destination_x: np.ndarray,
    destination_y: np.ndarray,
) -> tuple[pd.DataFrame, int]:
    output = pt_walk_default_records(origins["grid_id"])
    if destinations.empty:
        return output, 0

    radius_sq = PT_WALK_EUCLIDEAN_PREFILTER_M ** 2
    completed_requests = 0
    origin_grid_ids = origins["grid_id"].tolist()
    origin_x = origins["centroid_x_3035"].to_numpy(dtype=float)
    origin_y = origins["centroid_y_3035"].to_numpy(dtype=float)
    distance_sq = (origin_x[:, None] - destination_x[None, :]) ** 2 + (origin_y[:, None] - destination_y[None, :]) ** 2

    eligible_by_origin = {}
    pending_positions = []
    requested_destination_indices = set()
    for row_position, grid_id in enumerate(origin_grid_ids):
        candidate_indices = np.flatnonzero(distance_sq[row_position] <= radius_sq)
        if candidate_indices.size == 0:
            continue
        candidate_index_set = {int(index) for index in candidate_indices.tolist()}
        eligible_by_origin[grid_id] = candidate_index_set
        pending_positions.append(row_position)
        requested_destination_indices.update(candidate_index_set)

    if not requested_destination_indices:
        return output, 0

    pending_origins = origins.iloc[pending_positions].copy()
    has_pt_by_grid = {grid_id: False for grid_id in origin_grid_ids}
    departures_by_grid = {grid_id: 0.0 for grid_id in origin_grid_ids}
    destination_indices = sorted(requested_destination_indices)

    for destination_start in range(0, len(destination_indices), DESTINATION_CHUNK_SIZE):
        chunk_indices = destination_indices[destination_start:destination_start + DESTINATION_CHUNK_SIZE]
        destination_chunk = destinations.iloc[chunk_indices].copy()
        matrix = route_matrix(pending_origins, destination_chunk, costing="pedestrian")
        for origin_row, result_row in zip(pending_origins.itertuples(index=False), matrix):
            eligible_indices = eligible_by_origin.get(origin_row.grid_id, set())
            for destination_index, result in enumerate(result_row):
                destination_global_index = int(destination_chunk.index[destination_index])
                if destination_global_index not in eligible_indices:
                    continue
                if result.get("status", 0) != 0:
                    continue
                result_time = result.get("time")
                if not finite_number(result_time) or float(result_time) > PT_WALK_MAX_SECONDS:
                    continue
                departures = destination_chunk.iloc[destination_index].get("pt_departures_weekday")
                departures_value = 0.0 if pd.isna(departures) else float(departures)
                has_pt_by_grid[origin_row.grid_id] = True
                departures_by_grid[origin_row.grid_id] += departures_value
        completed_requests += 1

    output["has_pt_stop_5min_walk"] = output["grid_id"].map(has_pt_by_grid).fillna(False).astype(bool)
    output["pt_departures_5min_walk"] = output["grid_id"].map(departures_by_grid).fillna(0.0).astype(float)
    return output, completed_requests


def pt_walk_for_type(active_cells: pd.DataFrame, destinations: gpd.GeoDataFrame, progress_position: int | None = None) -> pd.DataFrame:
    if destinations.empty:
        return pt_walk_default_records(active_cells["grid_id"])

    destinations_3035 = destinations.to_crs("EPSG:3035")
    destination_x = destinations_3035.geometry.x.to_numpy(dtype=float)
    destination_y = destinations_3035.geometry.y.to_numpy(dtype=float)
    chunk_starts = list(range(0, len(active_cells), ORIGIN_CHUNK_SIZE))
    completed_requests = 0
    chunk_results = {}
    progress_kwargs = {
        "total": len(active_cells),
        "desc": "PT walk access",
        "unit": "origin",
        "leave": True,
    }
    if progress_position is not None:
        progress_kwargs["position"] = progress_position
    progress_bar = tqdm(**progress_kwargs)

    def submit_next(executor, next_chunk_index: int, futures: dict) -> int:
        if next_chunk_index >= len(chunk_starts):
            return next_chunk_index
        chunk_start = chunk_starts[next_chunk_index]
        origins = active_cells.iloc[chunk_start:chunk_start + ORIGIN_CHUNK_SIZE].copy()
        future = executor.submit(pt_walk_for_origin_chunk, origins, destinations, destination_x, destination_y)
        futures[future] = chunk_start
        return next_chunk_index + 1

    try:
        with ThreadPoolExecutor(max_workers=PT_WALK_ORIGIN_CHUNK_WORKERS) as executor:
            futures = {}
            next_chunk_index = 0
            while next_chunk_index < len(chunk_starts) and len(futures) < PT_WALK_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                next_chunk_index = submit_next(executor, next_chunk_index, futures)

            while futures:
                for completed in as_completed(list(futures)):
                    chunk_start = futures.pop(completed)
                    pt_chunk, request_count = completed.result()
                    chunk_results[chunk_start] = pt_chunk
                    completed_requests += request_count
                    progress_bar.update(len(pt_chunk))
                    progress_bar.set_postfix_str(f"{completed_requests:,} matrix requests")
                    while next_chunk_index < len(chunk_starts) and len(futures) < PT_WALK_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                        next_chunk_index = submit_next(executor, next_chunk_index, futures)
                    break
    finally:
        progress_bar.close()

    if not chunk_results:
        return pt_walk_default_records(active_cells["grid_id"])
    return pd.concat([chunk_results[chunk_start] for chunk_start in chunk_starts], ignore_index=True)


def merge_nearest_results(base_output: pd.DataFrame, destination_tables: dict, nearest_results_by_type: dict) -> pd.DataFrame:
    output = base_output.copy()
    for poi_type in [*DESTINATION_TYPES, PT_WALK_POI_TYPE]:
        destinations = destination_tables[poi_type]
        if destinations.empty:
            output[f"tt_{poi_type}_min"] = pd.NA
            output[f"km_{poi_type}"] = pd.NA
            output[f"nearest_{poi_type}_id"] = pd.NA
            output[f"routing_status_{poi_type}"] = "missing_destinations"
            continue
        nearest = nearest_results_by_type.get(poi_type)
        if nearest is None:
            continue
        output = output.merge(nearest, on="grid_id", how="left")
    return output


def merge_pt_walk_results(base_output: pd.DataFrame, pt_walk_results: pd.DataFrame | None) -> pd.DataFrame:
    output = base_output.copy()
    if pt_walk_results is None:
        return output
    output = output.drop(columns=["has_pt_stop_5min_walk", "pt_departures_5min_walk"], errors="ignore")
    return output.merge(pt_walk_results, on="grid_id", how="left")


def generate_nearest_infrastructure(year: int) -> Path:
    paths = output_paths(year)
    poi_path = POI_DIR / f"austria-{year}-pois.geoparquet"
    if not poi_path.exists():
        raise FileNotFoundError(f"Missing yearly POI file: {poi_path}")

    active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
    if MAX_ACTIVE_CELLS is not None:
        active_cells = active_cells.head(MAX_ACTIVE_CELLS).copy()

    pois = gpd.read_parquet(poi_path)
    base_output = active_cells[["grid_id"]].copy()
    base_output["has_pt_stop_5min_walk"] = False
    base_output["pt_departures_5min_walk"] = 0.0
    available_types = sorted(pois["poi_type"].dropna().unique()) if "poi_type" in pois.columns else []
    print(f"{year}: {len(active_cells):,} origins, POI types: {available_types}")

    destination_tables = {poi_type: normalize_destinations(pois, poi_type) for poi_type in [*DESTINATION_TYPES, PT_WALK_POI_TYPE]}
    nearest_results_by_type = {}
    non_empty_types = []
    for poi_type in DESTINATION_TYPES:
        destinations = destination_tables[poi_type]
        if destinations.empty:
            tqdm.write(f"Skip {poi_type}: no destinations in {poi_path.name}")
            continue
        non_empty_types.append(poi_type)

    if non_empty_types:
        type_positions = {poi_type: index for index, poi_type in enumerate(non_empty_types)}
        with ThreadPoolExecutor(max_workers=min(NEAREST_INFRA_TYPE_WORKERS, len(non_empty_types))) as executor:
            futures = {
                executor.submit(
                    nearest_for_type,
                    active_cells,
                    destination_tables[poi_type],
                    poi_type,
                    type_positions[poi_type],
                    "auto",
                    NEAREST_INFRA_EUCLIDEAN_PREFILTER_M,
                ): poi_type
                for poi_type in non_empty_types
            }
            for completed in as_completed(futures):
                poi_type = futures[completed]
                nearest_results_by_type[poi_type] = completed.result()
                checkpoint = merge_nearest_results(base_output, destination_tables, nearest_results_by_type)
                checkpoint["year"] = year
                checkpoint["created_at"] = datetime.now().isoformat(timespec="seconds")
                checkpoint.to_parquet(paths["nearest_partial"], index=False)
                tqdm.write(f"Checkpointed nearest infrastructure after {poi_type} to {paths['nearest_partial']}")

    pt_walk_results = None
    pt_destinations = destination_tables[PT_WALK_POI_TYPE]
    if pt_destinations.empty:
        tqdm.write(f"Skip {PT_WALK_POI_TYPE}: no destinations in {poi_path.name}")
    else:
        # Public-transport stop access is pedestrian routing. In addition to the
        # 5-minute service-frequency exposure, retain the closest routable stop.
        nearest_results_by_type[PT_WALK_POI_TYPE] = nearest_for_type(
            active_cells, pt_destinations, PT_WALK_POI_TYPE, len(non_empty_types),
            "pedestrian", PT_NEAREST_EUCLIDEAN_PREFILTER_M, PT_NEAREST_DESTINATION_STEP_SHARE,
        )
        pt_walk_results = pt_walk_for_type(active_cells, pt_destinations, progress_position=len(non_empty_types))
        checkpoint = merge_nearest_results(base_output, destination_tables, nearest_results_by_type)
        checkpoint = merge_pt_walk_results(checkpoint, pt_walk_results)
        checkpoint["year"] = year
        checkpoint["created_at"] = datetime.now().isoformat(timespec="seconds")
        checkpoint.to_parquet(paths["nearest_partial"], index=False)
        tqdm.write(f"Checkpointed nearest infrastructure after {PT_WALK_POI_TYPE} to {paths['nearest_partial']}")

    output = merge_nearest_results(base_output, destination_tables, nearest_results_by_type)
    output = merge_pt_walk_results(output, pt_walk_results)
    output["year"] = year
    output["created_at"] = datetime.now().isoformat(timespec="seconds")
    nearest_tmp_path = paths["nearest"].with_name(paths["nearest"].name + ".tmp")
    output.to_parquet(nearest_tmp_path, index=False)
    nearest_tmp_path.replace(paths["nearest"])
    if paths["nearest_partial"].exists():
        paths["nearest_partial"].unlink()
    print(f"Wrote {len(output):,} rows to {paths['nearest']}")
    return paths["nearest"]


def load_active_cells_for_run() -> pd.DataFrame:
    active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
    if MAX_ACTIVE_CELLS is not None:
        active_cells = active_cells.head(MAX_ACTIVE_CELLS).copy()
    return active_cells.reset_index(drop=True)

In [ ]:
def year_quarters(year: int) -> pd.PeriodIndex:
    return pd.period_range(f"{year}Q1", f"{year}Q4", freq="Q")


def accessibility_minutes() -> list[int]:
    return sorted(set(int(minutes) for minutes in POPULATION_ACCESS_CONTOURS_MIN))


def fachgruppe_stock_columns_local() -> list[str]:
    return fachgruppe_stock_columns(FACHGRUPPE_IDS)


def accessibility_output_columns() -> list[str]:
    columns = []
    for minutes in accessibility_minutes():
        columns.extend([
            f"pop_access_{minutes}min",
            f"existing_firms_access_{minutes}min",
            *[f"fachgruppe_{fachgruppe_id}_access_{minutes}min" for fachgruppe_id in FACHGRUPPE_IDS],
        ])
    return columns


def firm_accessibility_columns() -> list[str]:
    columns = []
    for minutes in accessibility_minutes():
        columns.extend([
            f"pop_access_{minutes}min",
            f"existing_firms_access_{minutes}min",
            f"same_fachgruppe_firms_access_{minutes}min",
        ])
    return columns


def quarter_table_for_year(year: int) -> pd.DataFrame:
    quarters = year_quarters(year)
    return pd.DataFrame({
        "year": quarters.year.astype(int),
        "quarter": quarters.quarter.astype(int),
        "period": quarters.astype(str),
    })


_raster_centroids_4326_cache: gpd.GeoDataFrame | None = None
_firm_accessibility_source_cache: pd.DataFrame | None = None


def load_yearly_accessibility_panel(year: int) -> pd.DataFrame:
    if not PANEL_PATH.exists():
        raise FileNotFoundError(f"Missing raster quarter panel file: {PANEL_PATH}")
    required_columns = [
        "grid_id",
        "year",
        "quarter",
        "period",
        "population_backcast",
        "active_firms_tminus1",
        *fachgruppe_stock_columns_local(),
    ]
    panel = pd.read_parquet(
        PANEL_PATH,
        columns=required_columns,
        filters=[("year", "==", year)],
    )
    numeric_columns = ["population_backcast", "active_firms_tminus1", *fachgruppe_stock_columns_local()]
    for column in numeric_columns:
        panel[column] = pd.to_numeric(panel[column], errors="coerce").fillna(0.0).astype(float)
    panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype(int)
    panel["quarter"] = pd.to_numeric(panel["quarter"], errors="coerce").astype(int)
    panel["period"] = panel["period"].astype(str)
    return panel


def load_raster_centroids_4326() -> gpd.GeoDataFrame:
    global _raster_centroids_4326_cache
    if _raster_centroids_4326_cache is not None:
        return _raster_centroids_4326_cache
    if not RASTER_PATH.exists():
        raise FileNotFoundError(f"Missing raster geometry file: {RASTER_PATH}")
    raster = gpd.read_parquet(RASTER_PATH, columns=["grid_id", "geometry"])
    raster["geometry"] = raster.geometry.centroid
    raster = raster.to_crs("EPSG:4326")[["grid_id", "geometry"]].set_index("grid_id")
    _raster_centroids_4326_cache = raster
    return _raster_centroids_4326_cache


def load_accessibility_destinations_for_year(panel_year: pd.DataFrame) -> gpd.GeoDataFrame:
    mass_columns = ["population_backcast", "active_firms_tminus1", *fachgruppe_stock_columns_local()]
    positive_grid_mask = panel_year.groupby("grid_id")[mass_columns].max().gt(0).any(axis=1)
    positive_grid_ids = positive_grid_mask.index[positive_grid_mask].tolist()
    if not positive_grid_ids:
        return gpd.GeoDataFrame(
            {"grid_id": pd.Series(dtype="object")},
            geometry=gpd.GeoSeries([], crs="EPSG:4326"),
            crs="EPSG:4326",
        )
    centroids = load_raster_centroids_4326()
    destinations = centroids.loc[centroids.index.intersection(positive_grid_ids)].reset_index()
    return gpd.GeoDataFrame(destinations, geometry="geometry", crs="EPSG:4326")


def accessibility_part_paths(parts_dir: Path) -> list[Path]:
    return sorted(parts_dir.glob("part_*.parquet"))


def accessibility_part_paths_match_schema(part_paths: list[Path]) -> bool:
    if not part_paths:
        return True
    required_columns = {"grid_id", "year", "quarter", "period"}
    sample = pd.read_parquet(part_paths[0])
    return required_columns.issubset(set(sample.columns))


def clear_accessibility_part_paths(part_paths: list[Path]) -> None:
    for part_path in part_paths:
        if part_path.exists():
            part_path.unlink()


def remove_accessibility_parts_dir(parts_dir: Path) -> None:
    part_paths = accessibility_part_paths(parts_dir)
    clear_accessibility_part_paths(part_paths)
    if parts_dir.exists() and not any(parts_dir.iterdir()):
        parts_dir.rmdir()


def completed_accessibility_origin_ids(part_paths: list[Path]) -> set:
    completed = set()
    for part_path in part_paths:
        part = pd.read_parquet(part_path, columns=["grid_id"])
        completed.update(part["grid_id"].dropna().unique())
    return completed


def accessibility_contour_minutes(feature: dict) -> int:
    properties = feature.get("properties", {})
    for key in ["contour", "time", "time_min", "time_minutes"]:
        value = properties.get(key)
        if finite_number(value):
            return int(round(float(value)))
        try:
            return int(round(float(value)))
        except (TypeError, ValueError):
            continue
    raise KeyError(f"Isochrone feature is missing a contour value in properties: {properties}")


def request_accessibility_isochrones(origin: pd.Series) -> dict[int, object]:
    payload = {
        "locations": [{"lat": float(origin.lat), "lon": float(origin.lon)}],
        "costing": POPULATION_ACCESS_COSTING,
        "contours": [{"time": minutes} for minutes in accessibility_minutes()],
        "polygons": POPULATION_ACCESS_POLYGONS,
        "denoise": POPULATION_ACCESS_DENOISE,
        "generalize": POPULATION_ACCESS_GENERALIZE_M,
    }
    desired_contours = accessibility_minutes()
    last_error = None
    for attempt in range(1, POPULATION_ACCESS_REQUEST_RETRIES + 1):
        try:
            response = requests.post(f"{VALHALLA_URL}/isochrone", json=payload, timeout=POPULATION_ACCESS_REQUEST_TIMEOUT_SECONDS)
            if not response.ok:
                raise RuntimeError(
                    f"Valhalla isochrone request failed with HTTP {response.status_code}: {response.text[:1000]}"
                )
            data = response.json()
            features = data.get("features", [])
            if not features:
                raise RuntimeError(f"Unexpected Valhalla isochrone response keys: {sorted(data.keys())}")
            contour_geometries = {}
            for feature in features:
                contour_minutes = accessibility_contour_minutes(feature)
                if contour_minutes not in desired_contours:
                    continue
                geometry_payload = feature.get("geometry")
                if geometry_payload is None:
                    continue
                geometry = shape(geometry_payload)
                if geometry.is_empty:
                    continue
                if contour_minutes in contour_geometries:
                    contour_geometries[contour_minutes] = contour_geometries[contour_minutes].union(geometry)
                else:
                    contour_geometries[contour_minutes] = geometry
            missing = [minutes for minutes in desired_contours if minutes not in contour_geometries]
            if missing:
                raise RuntimeError(f"Isochrone response for {origin.grid_id} is missing contours: {missing}")
            return contour_geometries
        except Exception as error:
            last_error = error
            if attempt == POPULATION_ACCESS_REQUEST_RETRIES:
                raise
            time.sleep(min(10, attempt * 2))
    raise RuntimeError(f"Isochrone request failed for {origin.grid_id}: {last_error}")


def reachable_grid_ids_by_contour(origin: pd.Series, destination_cells: gpd.GeoDataFrame) -> dict[int, list[str]]:
    contour_geometries = request_accessibility_isochrones(origin)
    reachable = {}
    for minutes in accessibility_minutes():
        contour_geometry = contour_geometries[minutes]
        reachable_grid_ids = {origin["grid_id"]}
        candidate_indices = destination_cells.sindex.query(box(*contour_geometry.bounds), predicate="intersects")
        if len(candidate_indices) > 0:
            candidates = destination_cells.iloc[candidate_indices]
            reachable_grid_ids.update(candidates.loc[candidates.geometry.intersects(contour_geometry), "grid_id"].tolist())
        reachable[minutes] = sorted(reachable_grid_ids)
    return reachable


def accessibility_records_for_origin(
    origin: pd.Series,
    year: int,
    quarterly_mass_tables: dict[int, pd.DataFrame],
    quarter_period_map: dict[int, str],
    destination_cells: gpd.GeoDataFrame,
) -> list[dict]:
    reachable_by_contour = reachable_grid_ids_by_contour(origin, destination_cells)
    records = []
    for quarter, period in quarter_period_map.items():
        mass_table = quarterly_mass_tables[quarter]
        record = {
            "grid_id": origin["grid_id"],
            "year": int(year),
            "quarter": int(quarter),
            "period": period,
        }
        for minutes, accessible_grid_ids in reachable_by_contour.items():
            accessible_masses = mass_table.reindex(accessible_grid_ids, fill_value=0.0)
            record[f"pop_access_{minutes}min"] = float(accessible_masses["population_backcast"].sum())
            record[f"existing_firms_access_{minutes}min"] = float(accessible_masses["active_firms_tminus1"].sum())
            for fachgruppe_id in FACHGRUPPE_IDS:
                source_column = f"fachgruppe_{fachgruppe_id}_active_firms_tminus1"
                target_column = f"fachgruppe_{fachgruppe_id}_access_{minutes}min"
                record[target_column] = float(accessible_masses[source_column].sum())
        records.append(record)
    return records


def write_accessibility_potentials_output(
    year: int,
    part_paths: list[Path],
    output_path: Path,
    active_cells: pd.DataFrame,
) -> int:
    output = active_cells[["grid_id"]].copy().merge(quarter_table_for_year(year), how="cross")
    if part_paths:
        combined = pd.concat([pd.read_parquet(part_path) for part_path in part_paths], ignore_index=True)
        combined = combined.drop_duplicates(["grid_id", "year", "quarter"], keep="last")
        combined = combined.drop(columns=["period"], errors="ignore")
        output = output.merge(combined, on=["grid_id", "year", "quarter"], how="left")
    for column in accessibility_output_columns():
        if column in output.columns:
            output[column] = pd.to_numeric(output[column], errors="coerce").fillna(0.0).astype(float)
        else:
            output[column] = 0.0
    output["created_at"] = datetime.now().isoformat(timespec="seconds")
    output = output[["grid_id", "year", "quarter", "period", *accessibility_output_columns(), "created_at"]]
    tmp_path = output_path.with_name(output_path.name + ".tmp")
    output.to_parquet(tmp_path, index=False)
    tmp_path.replace(output_path)
    return len(output)


def prepare_accessibility_potential_inputs(year: int, active_cells: pd.DataFrame | None = None) -> dict:
    if active_cells is None:
        active_cells = load_active_cells_for_run()
    panel_year = load_yearly_accessibility_panel(year)
    destination_cells = load_accessibility_destinations_for_year(panel_year).reset_index(drop=True)
    _ = destination_cells.sindex
    quarter_period_map = {int(period.quarter): str(period) for period in year_quarters(year)}
    quarterly_mass_tables = {}
    for quarter, period in quarter_period_map.items():
        quarter_frame = (
            panel_year.loc[panel_year["quarter"] == quarter, ["grid_id", "population_backcast", "active_firms_tminus1", *fachgruppe_stock_columns_local()]]
            .drop_duplicates("grid_id", keep="last")
            .set_index("grid_id")
        )
        quarterly_mass_tables[quarter] = quarter_frame
    return {
        "active_cells": active_cells.reset_index(drop=True).copy(),
        "quarterly_mass_tables": quarterly_mass_tables,
        "quarter_period_map": quarter_period_map,
        "destination_cells": destination_cells,
    }


def generate_accessibility_potentials(year: int, prepared_inputs: dict | None = None) -> Path:
    paths = output_paths(year)
    if prepared_inputs is None:
        prepared_inputs = prepare_accessibility_potential_inputs(year)
    active_cells = prepared_inputs["active_cells"].copy()
    quarterly_mass_tables = prepared_inputs["quarterly_mass_tables"]
    quarter_period_map = prepared_inputs["quarter_period_map"]
    destination_cells = prepared_inputs["destination_cells"]

    parts_dir = paths["potentials_parts"]
    parts_dir.mkdir(parents=True, exist_ok=True)
    part_paths = accessibility_part_paths(parts_dir)
    if part_paths and not accessibility_part_paths_match_schema(part_paths):
        print(
            "Detected legacy accessibility parts without quarter keys. "
            "Clearing incompatible part files and restarting the accessibility run for this year."
        )
        clear_accessibility_part_paths(part_paths)
        part_paths = []
    completed_origin_ids = completed_accessibility_origin_ids(part_paths)
    if completed_origin_ids:
        print(
            f"Resuming accessibility potentials from {len(part_paths):,} part files "
            f"with {len(completed_origin_ids):,} origins already represented"
        )

    records_buffer = []
    part_number = len(part_paths)

    def flush_records() -> None:
        nonlocal records_buffer, part_number
        if not records_buffer:
            return
        part_number += 1
        part_path = parts_dir / f"part_{part_number:05d}.parquet"
        part = pd.DataFrame(records_buffer)
        for column in accessibility_output_columns():
            part[column] = pd.to_numeric(part[column], errors="coerce").fillna(0.0).astype(float)
        part.to_parquet(part_path, index=False)
        part_paths.append(part_path)
        records_buffer = []
        gc.collect()

    def submit_next(executor, next_origin_index: int, futures: dict) -> int:
        while next_origin_index < len(active_cells):
            grid_id = active_cells["grid_id"].iloc[next_origin_index]
            if grid_id not in completed_origin_ids:
                origin = active_cells.iloc[next_origin_index].copy()
                future = executor.submit(
                    accessibility_records_for_origin,
                    origin,
                    year,
                    quarterly_mass_tables,
                    quarter_period_map,
                    destination_cells,
                )
                futures[future] = grid_id
                return next_origin_index + 1
            next_origin_index += 1
        return next_origin_index

    progress_bar = tqdm(
        total=len(active_cells),
        desc="Accessibility potentials",
        unit="origin",
        initial=len(completed_origin_ids),
        leave=True,
    )
    try:
        with ThreadPoolExecutor(max_workers=POPULATION_ACCESS_ORIGIN_WORKERS) as executor:
            futures = {}
            next_origin_index = 0
            while next_origin_index < len(active_cells) and len(futures) < POPULATION_ACCESS_MAX_IN_FLIGHT_ORIGINS:
                next_origin_index = submit_next(executor, next_origin_index, futures)

            while futures:
                for completed in as_completed(list(futures)):
                    grid_id = futures.pop(completed)
                    origin_records = completed.result()
                    if origin_records:
                        records_buffer.extend(origin_records)
                    completed_origin_ids.add(grid_id)
                    if len(records_buffer) >= POPULATION_ACCESS_FLUSH_ORIGINS * len(quarter_period_map):
                        flush_records()
                    progress_bar.update(1)
                    progress_bar.set_postfix_str(f"{len(completed_origin_ids):,} origins completed")
                    while next_origin_index < len(active_cells) and len(futures) < POPULATION_ACCESS_MAX_IN_FLIGHT_ORIGINS:
                        next_origin_index = submit_next(executor, next_origin_index, futures)
                    break
    finally:
        progress_bar.close()

    flush_records()
    if len(completed_origin_ids) != len(active_cells):
        raise RuntimeError(
            f"Accessibility potentials finished with {len(completed_origin_ids):,} completed origins, "
            f"expected {len(active_cells):,}"
        )
    total_rows = write_accessibility_potentials_output(year, part_paths, paths["potentials"], active_cells)
    print(f"Wrote {total_rows:,} rows to {paths['potentials']}")
    return paths["potentials"]


def load_firm_accessibility_source() -> pd.DataFrame:
    global _firm_accessibility_source_cache
    if _firm_accessibility_source_cache is not None:
        return _firm_accessibility_source_cache
    if not FIRMS_PATH.exists():
        raise FileNotFoundError(f"Missing firm assignment file: {FIRMS_PATH}")
    firms = pd.read_parquet(FIRMS_PATH, columns=["firm_id", "grid_id_100m", "Fachgruppe_ID", "founding_date", "exit_date"])
    firms = firms.dropna(subset=["firm_id", "grid_id_100m", "founding_date"]).copy()
    firms["founding_date"] = pd.to_datetime(firms["founding_date"], errors="coerce")
    firms["exit_date"] = pd.to_datetime(firms["exit_date"], errors="coerce")
    firms = firms.dropna(subset=["founding_date"]).copy()
    firms["exit_date_filled"] = firms["exit_date"].fillna(pd.Timestamp.max)
    firms["Fachgruppe_ID"] = firms["Fachgruppe_ID"].astype("string")
    firms["Fachgruppe_ID_normalized"] = firms["Fachgruppe_ID"]
    _firm_accessibility_source_cache = firms
    return _firm_accessibility_source_cache


def build_firm_quarter_panel_for_year(year: int) -> pd.DataFrame:
    firms = load_firm_accessibility_source()
    records = []
    for period in year_quarters(year):
        quarter_end = period.end_time.normalize()
        previous_quarter_end = (period - 1).end_time.normalize()
        active_in_period = firms[
            (firms["founding_date"] <= quarter_end)
            & (firms["exit_date_filled"] > previous_quarter_end)
        ].copy()
        if active_in_period.empty:
            continue
        active_in_period["year"] = int(period.year)
        active_in_period["quarter"] = int(period.quarter)
        active_in_period["period"] = str(period)
        active_in_period["included_in_lagged_stock"] = (
            (active_in_period["founding_date"] <= previous_quarter_end)
            & (active_in_period["exit_date_filled"] > previous_quarter_end)
        )
        records.append(
            active_in_period[[
                "firm_id",
                "grid_id_100m",
                "Fachgruppe_ID",
                "Fachgruppe_ID_normalized",
                "year",
                "quarter",
                "period",
                "included_in_lagged_stock",
            ]]
        )
    if not records:
        return pd.DataFrame(columns=[
            "firm_id",
            "grid_id_100m",
            "Fachgruppe_ID",
            "Fachgruppe_ID_normalized",
            "year",
            "quarter",
            "period",
            "included_in_lagged_stock",
        ])
    return pd.concat(records, ignore_index=True)


def generate_firm_accessibility_output(year: int, cell_accessibility_path: Path) -> Path:
    paths = output_paths(year)
    if not cell_accessibility_path.exists():
        raise FileNotFoundError(f"Missing cell accessibility output: {cell_accessibility_path}")
    cell_accessibility = pd.read_parquet(cell_accessibility_path).drop(columns=["created_at"], errors="ignore")
    cell_accessibility = cell_accessibility.rename(columns={"grid_id": "grid_id_100m"})
    firm_panel = build_firm_quarter_panel_for_year(year)
    joined = firm_panel.merge(
        cell_accessibility,
        on=["grid_id_100m", "year", "quarter", "period"],
        how="left",
    )
    for column in accessibility_output_columns():
        if column in joined.columns:
            joined[column] = pd.to_numeric(joined[column], errors="coerce").fillna(0.0).astype(float)
        else:
            joined[column] = 0.0
    for minutes in accessibility_minutes():
        same_fachgruppe_column = f"same_fachgruppe_firms_access_{minutes}min"
        joined[same_fachgruppe_column] = 0.0
        for fachgruppe_id in FACHGRUPPE_IDS:
            source_column = f"fachgruppe_{fachgruppe_id}_access_{minutes}min"
            mask = joined["Fachgruppe_ID_normalized"] == fachgruppe_id
            joined.loc[mask, same_fachgruppe_column] = joined.loc[mask, source_column]
    lagged_mask = joined["included_in_lagged_stock"].fillna(False).astype(bool)
    same_fachgruppe_mask = lagged_mask & joined["Fachgruppe_ID_normalized"].isin(FACHGRUPPE_IDS)
    for minutes in accessibility_minutes():
        existing_column = f"existing_firms_access_{minutes}min"
        same_fachgruppe_column = f"same_fachgruppe_firms_access_{minutes}min"
        joined.loc[lagged_mask, existing_column] = np.maximum(joined.loc[lagged_mask, existing_column] - 1.0, 0.0)
        joined.loc[same_fachgruppe_mask, same_fachgruppe_column] = np.maximum(joined.loc[same_fachgruppe_mask, same_fachgruppe_column] - 1.0, 0.0)
    output_columns = [
        "firm_id",
        "grid_id_100m",
        "Fachgruppe_ID",
        "year",
        "quarter",
        "period",
        "included_in_lagged_stock",
        *firm_accessibility_columns(),
    ]
    output = joined[output_columns].copy()
    output["created_at"] = datetime.now().isoformat(timespec="seconds")
    tmp_path = paths["firm_accessibility"].with_name(paths["firm_accessibility"].name + ".tmp")
    output.to_parquet(tmp_path, index=False)
    tmp_path.replace(paths["firm_accessibility"])
    print(f"Wrote {len(output):,} firm-quarter rows to {paths['firm_accessibility']}")
    return paths["firm_accessibility"]


def split_fachgruppe_accessibility(wide_path: Path, long_path: Path) -> None:
    """Stream the internal wide result into the two supported model products."""
    wide_columns = fachgruppe_access_columns(FACHGRUPPE_IDS)
    key_columns = ["grid_id", "year", "quarter"]
    main_columns = [*key_columns, "period", *main_access_columns(), "created_at"]
    long_writer = None
    main_writer = None
    main_tmp = wide_path.with_name(wide_path.name + ".narrow.tmp")
    long_tmp = long_path.with_name(long_path.name + ".tmp")
    try:
        parquet = pq.ParquetFile(wide_path)
        for batch in parquet.iter_batches(batch_size=5_000):
            frame = batch.to_pandas()
            main_table = pa.Table.from_pandas(frame[main_columns], preserve_index=False)
            if main_writer is None:
                main_writer = pq.ParquetWriter(main_tmp, main_table.schema, compression="snappy")
            main_writer.write_table(main_table)
            for fachgruppe_id in FACHGRUPPE_IDS:
                long_frame = frame[key_columns].copy()
                long_frame["Fachgruppe_ID"] = fachgruppe_id
                for minutes in ACCESS_MINUTES:
                    long_frame[f"same_fachgruppe_firms_access_{minutes}min"] = frame[f"fachgruppe_{fachgruppe_id}_access_{minutes}min"].to_numpy()
                long_table = pa.Table.from_pandas(long_frame, preserve_index=False)
                if long_writer is None:
                    long_writer = pq.ParquetWriter(long_tmp, long_table.schema, compression="snappy")
                long_writer.write_table(long_table)
    finally:
        if 'parquet' in locals(): parquet.close()
        if main_writer: main_writer.close()
        if long_writer: long_writer.close()
    main_tmp.replace(wide_path)
    long_tmp.replace(long_path)

## Run

The single cell below performs preflight, starts one yearly graph at a time, records concise status rows, and stops the container even after an error.

In [ ]:
if RUN_MODE not in {"dry-run", "smoke", "full"}:
    raise ValueError("RUN_MODE must be dry-run, smoke, or full")
for path in [ACTIVE_CELLS_PATH, PANEL_PATH, FIRMS_PATH, RASTER_PATH]:
    if not path.exists(): raise FileNotFoundError(path)
pd.DataFrame({"setting": ["mode", "years", "Fachgruppen", "firm mass"], "value": [RUN_MODE, str(YEARS_TO_RUN), len(FACHGRUPPE_IDS), "active_firms_tminus1"]})

In [ ]:
status_rows = []
for year in YEARS_TO_RUN:
    paths = output_paths(year)
    print(pd.DataFrame({"product": paths.keys(), "path": map(str, paths.values())}).to_string(index=False))
    if RUN_MODE == "dry-run":
        continue
    require_wsl_distribution()
    start_valhalla_container(year)
    try:
        wait_until_valhalla_ready()
        if RUN_MODE == "smoke":
            print({"year": year, "graz_route": valhalla_test_route(), "writes": 0})
            continue
        if RUN_NEAREST_INFRASTRUCTURE:
            output = generate_nearest_infrastructure(year)
            status_rows.append({"year": year, "product": "nearest_infrastructure_100m", "status": "done", "output": str(output)})
        if RUN_ACCESSIBILITY:
            wide = generate_accessibility_potentials(year)
            firm = generate_firm_accessibility_output(year, wide)
            split_fachgruppe_accessibility(wide, paths["fachgruppe_accessibility"])
            status_rows.extend([
                {"year": year, "product": "accessibility_potentials_100m", "status": "done", "output": str(wide), "firm_mass_source": "active_firms_tminus1"},
                {"year": year, "product": "fachgruppe_accessibility_quarter_100m", "status": "done", "output": str(paths["fachgruppe_accessibility"]), "firm_mass_source": "active_firms_tminus1"},
                {"year": year, "product": "firm_accessibility_quarter_100m", "status": "done", "output": str(firm), "firm_mass_source": "active_firms_tminus1"},
            ])
    finally:
        stop_valhalla_container(year)
if status_rows:
    pd.DataFrame(status_rows).to_csv(ROUTING_STATUS_PATH, index=False)
pd.DataFrame(status_rows)